# 🟦 Lab 04 · Meet PushT: explore a real robot dataset like a researcher

**World Models course · Capstone preparation (before C1) · connects lectures 5, 6, 12, 18, 23** &nbsp;|&nbsp; ⏱ about 60 min &nbsp;|&nbsp; 💻 CPU is enough (the optional DINOv2 section is faster on a GPU)

**PushT** is one of robot learning's best-known benchmarks, from Columbia and Toyota Research's *Diffusion Policy* paper. A round pusher (blue) must shove a T-shaped block (grey) until it covers a green T-shaped target. The dataset has **206 human demonstrations** with camera images, positions and actions.

In this lab you'll do what researchers do *before* training anything:

1. Open the data and **watch** demonstrations.
2. Work out what every number **means** (units, coordinates, what an "action" is).
3. Lock an honest **train / validation / test split**.
4. Build **baselines** that any fancy model must beat.
5. Ask: *can a vision encoder see where the block is?* Compare raw pixels with Meta's **DINOv2** features, global vs spatial.

This notebook prepares you for the **course capstone (C1–C4)**, where you train robot policies and a world model on this dataset. Section 7 also describes an optional research extension to explore after finishing the capstone.

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import importlib.util, subprocess, sys, json, hashlib, time, urllib.request
from pathlib import Path
if importlib.util.find_spec("zarr") is None or not __import__("zarr").__version__.startswith("2."):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "zarr==2.18.7", "numcodecs==0.15.1"])
    print("Installed zarr 2. If the next cell complains about zarr, use Runtime → Restart session and run again.")
import zarr
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
plt.rcParams.update({"figure.dpi": 110})

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
CHALLENGES["starts"] = dict(title="Where does each episode start?",
    reference=lambda ends: np.r_[0, ends[:-1]],
    cases=[np.array([3, 7, 12])],
    hint="The first episode starts at frame 0. Every other episode starts where the previous one ended: <code>np.r_[0, ends[:-1]]</code>.",
    why="All 25,650 frames are stored end to end. <code>episode_ends</code> is the only thing telling you where one demonstration stops and the next begins.")

CHALLENGES["to_pixels"] = dict(title="World units → image pixels",
    reference=lambda xy: xy * 96 / 512,
    cases=[np.array([[256., 256.], [0., 512.]])],
    hint="The simulator is 512 units wide; the image is 96 pixels wide. Multiply by 96 and divide by 512.",
    why="Mixing coordinate systems is the #1 silent bug in robot learning. Always write down the units.")

CHALLENGES["angle_diff"] = dict(title="Angle difference that wraps around",
    reference=lambda a, b: np.arctan2(np.sin(a - b), np.cos(a - b)),
    cases=[(np.array([0.1, 6.2, 3.0]), np.array([6.2, 0.1, 3.1]))],
    hint="Take the raw difference, then wrap it into (−π, π] with <code>np.arctan2(np.sin(d), np.cos(d))</code>.",
    why="359° and 1° are 2° apart, not 358°. Forgetting this makes a model look terrible at orientation for no real reason.")

CHALLENGES["downsample"] = dict(title="Shrink the image",
    reference=lambda images: images[:, ::8, ::8],
    cases=[np.arange(2 * 96 * 96 * 3).reshape(2, 96, 96, 3)],
    hint="Keep every 8th row and every 8th column, for all images and all colour channels: <code>images[:, ::8, ::8]</code>.",
    why="96×96 → 12×12. A tiny, transparent “encoder” to compare against a large pretrained one.")

QUIZZES["action"] = dict(predict=True, q="In PushT, the recorded <b>action</b> is two numbers. What do they most likely mean?",
    options=["The pusher's velocity", "A target position the pusher is pulled toward", "The force applied to the block"],
    answer=1, explain="The action is where the operator's cursor was. A PD controller pulls the pusher toward that point. It's in the same 0–512 units as positions, and usually a little ahead of the pusher.")
QUIZZES["still"] = dict(predict=True, q="In what fraction of time steps does the T-block move noticeably (more than 1 unit)?",
    options=["Almost always (>90%)", "Roughly a third of the time", "Rarely (<10%)"],
    answer=1, explain="The pusher spends a lot of time repositioning without touching the block. This makes “the block doesn't move” a strong baseline, and it means block-motion errors come from a minority of frames.")
QUIZZES["pooling_pusht"] = dict(q="Which DINOv2 representation should keep the block's <b>position</b> best?",
    options=["Global mean over all patches", "A 2×2 grid of averaged patches", "They must be identical"],
    answer=1, explain="Averaging all patches mixes <i>where</i> information together (lab 02). A coarse grid keeps some layout.")
print('✅ Setup complete. Scroll down and run the cells in order.')

---
## 1 · Download and open the data 📦
The archive is **31 MB** (from the official Diffusion Policy project). Inside is a *Zarr* store: a folder of compressed arrays that can be read piece by piece.

In [ ]:
DATA_URL = "https://diffusion-policy.cs.columbia.edu/data/training/pusht.zip"
archive = Path("pusht.zip")
if not archive.exists():
    print("Downloading PushT (31 MB)…"); urllib.request.urlretrieve(DATA_URL, archive)
store = zarr.ZipStore(str(archive), mode="r")
root = zarr.open_group(store=store, mode="r", path="pusht/pusht_cchi_v7_replay.zarr")
print(root.tree())

state  = root["data/state"][:].astype(np.float64)    # (frames, 5)
action = root["data/action"][:].astype(np.float64)   # (frames, 2)
ends   = root["meta/episode_ends"][:]                # (206,)
images = np.empty((len(state), 96, 96, 3), np.uint8)  # store images compactly as bytes (0–255)
for i in range(0, len(state), 2048):                  # decode in chunks to keep memory low
    images[i:i + 2048] = root["data/img"][i:i + 2048]
print(f"{len(ends)} episodes · {len(state)} frames · images use {images.nbytes / 1e9:.2f} GB of RAM")

| Array | Shape | Meaning |
|---|---|---|
| `images` | (25650, 96, 96, 3) | a small top-down RGB camera view |
| `state` | (25650, 5) | pusher x, pusher y, block x, block y, block angle (radians) |
| `action` | (25650, 2) | the commanded pusher target (x, y) |
| `ends` | (206,) | the frame index where each episode *ends* (exclusive) |

### 🧩 Challenge 1 · Where does each episode start?

In [ ]:
def episode_starts(ends):
    return ___     # 🧩 0 for the first episode, then each previous end

episode_starts = check("starts", episode_starts)
starts = episode_starts(ends)
lengths = ends - starts
print(f"episode length: shortest {lengths.min()}, average {lengths.mean():.0f}, longest {lengths.max()} steps (10 steps = 1 second)")

<details><summary>🤔 <b>Need a hint?</b></summary>

Prepend a 0 and drop the last end.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return np.r_[0, ends[:-1]]     # 🧩 0 for the first episode, then each previous end</pre>

</details>

## 2 · Watch a demonstration 🎬
### 🧩 Challenge 2 · World units → image pixels
Positions live in a 512×512 world. The image is 96×96. To draw the action on the image, convert it.

In [ ]:
def to_pixels(xy):
    return ___          # 🧩 rescale 0–512 → 0–96

to_pixels = check("to_pixels", to_pixels)

<details><summary>🤔 <b>Need a hint?</b></summary>

Scale by the ratio of image size to world size.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return xy * 96 / 512          # 🧩 rescale 0–512 → 0–96</pre>

</details>

### 🎛️ Playground · Scrub through any episode
The red ✕ is the **action**: where the pusher is being pulled.

In [ ]:
def watch(episode=0, progress=0.0):
    i = int(starts[episode] + progress * (lengths[episode] - 1))
    fig, axs = plt.subplots(1, 2, figsize=(9, 4))
    axs[0].imshow(images[i]); ax, ay = to_pixels(action[i]); axs[0].plot(ax, ay, "rx", ms=12, mew=3)
    axs[0].set_title(f"episode {episode} · step {i - starts[episode]} / {lengths[episode] - 1}"); axs[0].axis("off")
    seg = slice(starts[episode], ends[episode])
    axs[1].plot(state[seg, 0], state[seg, 1], c="tab:blue", alpha=0.6, label="pusher path")
    axs[1].plot(state[seg, 2], state[seg, 3], c="tab:gray", lw=3, label="block path")
    axs[1].plot(*state[i, :2], "o", c="tab:blue"); axs[1].plot(*state[i, 2:4], "s", c="k")
    axs[1].set(xlim=(0, 512), ylim=(512, 0), title="top view in world units"); axs[1].legend(fontsize=8); axs[1].set_aspect("equal")
    plt.tight_layout(); plt.show()
    print(f"state  = pusher ({state[i, 0]:.0f}, {state[i, 1]:.0f}) · block ({state[i, 2]:.0f}, {state[i, 3]:.0f}) · angle {state[i, 4]:.2f} rad")
    print(f"action = target ({action[i, 0]:.0f}, {action[i, 1]:.0f})")

playground(watch, episode=(0, 205, 1, 0), progress=(0.0, 1.0, 0.02, 0.0))

---
## 3 · What does the data really mean? 🔎

In [ ]:
quiz("action")

In [ ]:
gap_now  = np.linalg.norm(action - state[:, :2], axis=1)               # target vs where the pusher IS
nxt = np.arange(len(state) - 1); nxt = nxt[~np.isin(nxt, ends - 1)]   # skip the last frame of each episode
gap_next = np.linalg.norm(action[nxt] - state[nxt + 1, :2], axis=1)    # target vs where the pusher is NEXT step
print(f"median distance · action to current pusher position: {np.median(gap_now):.1f} units")
print(f"median distance · action to NEXT pusher position:    {np.median(gap_next):.1f} units")
print("→ the action is a target the pusher moves toward, not a velocity.")

In [ ]:
quiz("still")

In [ ]:
block_step = np.linalg.norm(state[nxt + 1, 2:4] - state[nxt, 2:4], axis=1)
print(f"block moves more than 1 unit in {np.mean(block_step > 1):.0%} of steps")

### 🧩 Challenge 3 · Angle difference that wraps around
The block angle lives in [0, 2π). Measuring orientation error needs care.

In [ ]:
def angle_difference(a, b):
    d = a - b
    return ___     # 🧩 wrap into (−π, π]

angle_difference = check("angle_diff", angle_difference)
print("6.2 rad vs 0.1 rad differ by", round(abs(float(angle_difference(6.2, 0.1))), 3), "rad, not 6.1")

<details><summary>🤔 <b>Need a hint?</b></summary>

Sine and cosine don't care about full turns; arctan2 turns them back into an angle.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return np.arctan2(np.sin(d), np.cos(d))     # 🧩 wrap into (−π, π]</pre>

</details>

---
## 4 · Lock the split before touching any model 🔒
Whole episodes only (lab 03). We save the split to a file so every later experiment, including the capstone, uses **exactly** the same test episodes.

In [ ]:
rng = np.random.default_rng(42)
order = rng.permutation(len(ends))
train_eps, val_eps, test_eps = order[:144], order[144:175], order[175:]
split = {"seed": 42, "train": train_eps.tolist(), "val": val_eps.tolist(), "test": test_eps.tolist(),
         "dataset_url": DATA_URL, "archive_sha256": hashlib.sha256(archive.read_bytes()).hexdigest()}
Path("pusht_split.json").write_text(json.dumps(split, indent=1))
print(f"train {len(train_eps)} · val {len(val_eps)} · test {len(test_eps)} episodes → saved pusht_split.json")

def frames_of(eps, horizon=0, stride=1):
    return np.concatenate([np.arange(starts[e], ends[e] - horizon, stride) for e in eps])

---
## 5 · Baselines: what must a model beat? 📏
We predict where the **block** will be `h` steps later from the current state and the next `h` actions.
* **Persistence:** "the block stays put."
* **Linear model:** ridge regression on (state, `sin`/`cos` of the angle, the next `h` actions).

We score by **episode** first, then average (lab 03).

In [ ]:
def features(i, h):
    s = state[i]
    acts = np.concatenate([action[i + k] for k in range(h)], axis=1) / 512
    return np.c_[s[:, :4] / 512, np.sin(s[:, 4]), np.cos(s[:, 4]), acts]

def block_errors(pred_xy, pred_angle, i, h):
    pos = np.linalg.norm(pred_xy - state[i + h, 2:4], axis=1)
    ang = np.abs(angle_difference(pred_angle, state[i + h, 4]))
    return pos, ang

rows = []
for h in [1, 5, 10]:
    tr = frames_of(train_eps, h)
    target = np.c_[state[tr + h, 2:4] - state[tr, 2:4], np.sin(state[tr + h, 4]), np.cos(state[tr + h, 4])]
    model = Ridge(alpha=1.0).fit(features(tr, h), target)
    per_ep = {"persistence": [], "linear model": []}
    for e in test_eps:
        i = frames_of([e], h)
        p = model.predict(features(i, h))
        for name, xy, ang in [("persistence", state[i, 2:4], state[i, 4]),
                              ("linear model", state[i, 2:4] + p[:, :2], np.arctan2(p[:, 2], p[:, 3]))]:
            pos, a = block_errors(xy, ang, i, h)
            per_ep[name].append((pos.mean(), np.degrees(a).mean()))
    for name, v in per_ep.items():
        v = np.array(v); rows.append((h, name, v[:, 0].mean(), v[:, 1].mean()))
print(f"{'h':>3} | {'baseline':>12} | block position error (units) | angle error (deg)")
for h, name, pos, ang in rows:
    print(f"{h:>3} | {name:>12} | {pos:28.2f} | {ang:17.2f}")

**What to notice:** the linear model does **not** beat "the block stays put" at 1 or 5 steps, and only barely at 10.

Why? The block moves only when the pusher *touches* it, and contact is sharply **nonlinear**: a push 1 unit to the left might miss entirely. A straight-line model can't represent that. This is precisely why robot learning uses neural world models and policies, and why **every result you report must be compared with persistence**.

---
## 6 · Can an encoder *see* the block? 👁️
A **probe** (lab 02) reads block position (x, y) and orientation (sin, cos) from image features. We compare:
1. **Raw pixels**, shrunk to 12×12.
2. **DINOv2** (Meta's self-supervised vision transformer): a global average of its patch features vs a 2×2 grid.

### 🧩 Challenge 4 · Shrink the image

In [ ]:
def downsample(images):
    return ___      # 🧩 every 8th row and column

downsample = check("downsample", downsample)

probe_frames = {name: frames_of(eps, stride=10) for name, eps in [("train", train_eps), ("test", test_eps)]}
def probe_target(i):
    return np.c_[state[i, 2:4] / 512, np.sin(state[i, 4]), np.cos(state[i, 4])]

def probe_report(name, feat_train, feat_test, alpha=1.0):
    tr, te = probe_frames["train"], probe_frames["test"]
    p = Ridge(alpha=alpha).fit(feat_train, probe_target(tr)).predict(feat_test)
    pos = np.linalg.norm(p[:, :2] * 512 - state[te, 2:4], axis=1).mean()
    ang = np.degrees(np.abs(angle_difference(np.arctan2(p[:, 2], p[:, 3]), state[te, 4]))).mean()
    print(f"{name:>28}: block position error {pos:6.1f} units · angle error {ang:5.1f}°  ({feat_train.shape[1]} numbers)")
    return pos, ang

pix = lambda i: downsample(images[i]).reshape(len(i), -1) / 255.0
results = {"raw pixels 12×12": probe_report("raw pixels 12×12", pix(probe_frames["train"]), pix(probe_frames["test"]), alpha=10.0)}
print(f"(for scale: always guessing the average position gives ≈ {np.linalg.norm(state[probe_frames['test'], 2:4] - state[probe_frames['train'], 2:4].mean(0), axis=1).mean():.0f} units)")

<details><summary>🤔 <b>Need a hint?</b></summary>

Slicing with a step: <code>start:stop:step</code>.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return images[:, ::8, ::8]      # 🧩 every 8th row and column</pre>

</details>

### Optional · DINOv2 features (about 2 minutes on CPU, seconds on a GPU)
DINOv2 splits a 224×224 image into a 16×16 grid of patches and returns a 384-number feature vector per patch. We then either **average all 256 patches** or **average into a 2×2 grid**.

> The first run downloads the small DINOv2 model (~85 MB) from Meta's official repository.

In [ ]:
RUN_DINO = True     # set to False to skip
if RUN_DINO:
    import torch, torch.nn.functional as F
    device = "cuda" if torch.cuda.is_available() else "cpu"
    encoder = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", trust_repo=True).eval().to(device)
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    @torch.inference_mode()
    def dino_features(frame_ids, batch=64):
        glob, grid = [], []
        for k in range(0, len(frame_ids), batch):
            x = torch.from_numpy(images[frame_ids[k:k + batch]]).to(device).permute(0, 3, 1, 2).float() / 255
            x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)      # 96 → 224 pixels
            tokens = encoder.forward_features((x - mean) / std)["x_norm_patchtokens"]  # (B, 256, 384)
            B, N, D = tokens.shape
            g = tokens.transpose(1, 2).reshape(B, D, 16, 16)                           # back into a 16×16 grid
            glob.append(tokens.mean(1).cpu().numpy())                                  # global average
            grid.append(F.adaptive_avg_pool2d(g, 2).flatten(1).cpu().numpy())          # 2×2 grid
        return np.concatenate(glob), np.concatenate(grid)

    t0 = time.time()
    g_tr, s_tr = dino_features(probe_frames["train"]); g_te, s_te = dino_features(probe_frames["test"])
    print(f"extracted {len(g_tr) + len(g_te)} frames in {time.time() - t0:.0f} s on {device}")

In [ ]:
quiz("pooling_pusht")

In [ ]:
if RUN_DINO:
    results["DINOv2 global average"] = probe_report("DINOv2 global average", g_tr, g_te, alpha=1.0)
    results["DINOv2 2×2 grid"] = probe_report("DINOv2 2×2 grid", s_tr, s_te, alpha=1.0)
    names = list(results); pos = [results[n][0] for n in names]
    plt.figure(figsize=(6, 2.6)); plt.barh(names, pos, color=["#bbb", "#f2a65a", "#3b8ea5"][:len(names)])
    plt.xlabel("block position error (world units, lower is better)"); plt.title("What can a linear probe read?"); plt.show()

⚠️ **Read carefully:** a probe tells you what a representation makes *easy to read out linearly*. It doesn't prove that information is absent, and it is **not** a world model. Predicting the *future* is the next step.

---
## 7 · Continue to the course capstone 🧭

Complete **C1 → C2 → C3 → C4**, starting with `capstone/C1_Flow_Matching_Policy_PushT.ipynb`. Build a flow-matching policy, a vision policy, a latent world model, and a fine-tuned SmolVLA. Submit one combined capstone report and code package.

Reuse `pusht_split.json`. Download it from the Colab file panel before your session ends.

### Optional research extension · after the capstone
**Question:** Which frozen visual representation best supports action-conditioned prediction of the block's future?
1. Compare DINOv2 global versus 2×2 spatial features with one versus two input frames.
2. Train a small predictor and probe predicted features for block position.
3. Compare persistence, the linear state baseline, and a shuffled-actions control.
4. Evaluate block position error at horizon 5, scored by episode with bootstrap intervals and three seeds.

This investigation is optional follow-on research; it is not a second capstone.


In [ ]:
progress_report()